In [0]:
# instalando bibliotecas para fuzzy matching
%pip install fuzzywuzzy python-Levenshtein

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

# LENDO OS DADOS DA CAMADA BRONZE
df_manobras = spark.table("workspace.default.bronze_manobras")
df_alarmes = spark.table("workspace.default.bronze_alarmes")

# ==========================================
# TRANSFORMANDO DADOS DE MANOBRAS
# ==========================================

# Convertendo datas de string para timestamp (Nomes corrigidos conforme o schema real)
date_cols = ['DT_INI_DESLOC', 'DT_INI_OPER', 'DT_FIM_OPER', 'DT_FIM_DESLOC']
for col_name in date_cols:
    df_manobras = df_manobras.withColumn(col_name, F.to_timestamp(F.col(col_name)))

# Removendo registros sem navio e removendo duplicatas
df_manobras = df_manobras.filter(F.col('NAVIO').isNotNull()).dropDuplicates()

# Ajustando nomes dos portos
de_para_portos = {
    'SANTOS': 'Porto de Santos/SP', 'RIO DE JANEIRO': 'Porto do Rio de Janeiro/RJ',
    'PARANAGUA': 'Porto de Paranaguá/PR', 'PONTA UBU': 'Porto de Ubú/ES',
    'ITAGUAI': 'Porto de Itaguai/RJ', 'SÃO SEBASTIÃO': 'Porto de São Sebastião/SP',
    'ITAQUI': 'Porto de São Luis/MA', 'RECIFE': 'Porto de Recife/PE',
    'ITAJAI': 'Porto de Itajai/SC', 'MACEIO': 'Porto de Maceió/AL',
    'VILA DO CONDE': 'Porto de Vila do Conde/PA', 'RIO GRANDE': 'Porto de Rio Grande/RS',
    'PONTA DA MADEIRA': 'Porto de São Luis/MA', 'ARATU': 'Porto de Salvador/BA',
    'TROMBETAS': 'Porto de Trombetas/PA', 'SALVADOR': 'Porto de Salvador/BA',
    'PORTO DO AÇU': 'Porto do Açú/RJ', 'SUAPE': 'Porto de Ipojuca/PE',
    'SAO FRANCISCO DO SUL': 'Porto de São Francisco do Sul/SC',
    'TEMADRE (MADRE DE DEUS)': 'Porto de Salvador/BA',
    'BARRA DO RIACHO': 'Porto de Barra do Riacho/ES', 'CABEDELO': 'Porto de Cabedelo/PB',
    'ILHEUS': 'Porto de Ilheus/BA', 'FORTALEZA': 'Porto de Mucuripe/CE',
    'IMBITUBA': 'Porto de Imbituba/SC', 'BELEM': 'Porto de Belém/PA',
    'NAVEGANTES': 'Porto de Itajai/SC', 'VITORIA': 'Porto de Vitoria/ES',
    'PRAIA MOLE': 'Porto de Vitoria/ES', 'PECEM': 'Porto de Pecém/CE',
    'TUBARÃO': 'Porto de Vitoria/ES', 'ITAPOA': 'Porto de São Francisco do Sul/SC',
    'BARRA DOS COQUEIROS': 'Porto de Barra dos Coqueiros/SE', 'SANTAREM': 'Porto de Santarém/PA',
    'SÃO ROQUE': 'Porto de Salvador/BA', 'ANGRA DOS REIS': 'Porto de Itaguai/RJ',
    'ALUMAR': 'Porto de São Luis/MA', 'ANTONINA': 'Porto de Paranaguá/PR'
}

df_manobras = df_manobras.withColumn(
    "PORTO",
    F.coalesce(F.create_map([F.lit(x) for x in sum(de_para_portos.items(), ())]).getItem(F.col("PORTO")), F.col("PORTO"))
)

# ==========================================
# TRANSFORMANDO DADOS ALARMES
# ==========================================

# Lista de rebocadores que iremos analisar 
rebocadores_ws = [
    'ALPHARD', 'ANDROMEDA', 'AQUARIUS', 'ATLAS', 'ATRIA', 'AURIGA', 'BAYOVAR',
    'CARINA', 'CAUE', 'CEPHEUS', 'CETUS', 'CNL SAFIRA', 'CNL-RUBI', 'CORONA',
    'CRATER', 'DELPHINUS', 'DRACO', 'ERIDANUS', 'FORNAX', 'HADAR', 'HAMAL',
    'HARIS', 'HERCULES', 'HUMBERTO', 'LACERTA', 'LYRA', 'MARTE', 'MERCURIUS',
    'NEPTUNO', 'OCTANS', 'OMEGA', 'PICTOR', 'PLUTAO', 'POLLUX II', 'PRAIA MOLE',
    'REGULUS', 'SAO LUIS', 'SATURNO', 'SCULPTOR', 'SEXTANS', 'SOSSEGO VALE',
    'SVITZER ELIAS D.', 'TAURUS1', 'TELESCOPIUM', 'URANUS', 'URSA', 'VEGA',
    'VELA', 'VITORIA LX', 'VOLANS', 'WEZEN', 'WS ANTARES', 'WS AQUILA',
    'WS ARCTURUS', 'WS ARIES', 'WS BELLATRIX', 'WS CASTOR', 'WS CENTAURUS',
    'WS CYGNUS', 'WS DORADO', 'WS GEMINI', 'WS HYDRUS', 'WS ITAQUI', 'WS JUPITER',
    'WS LEONIS', 'WS LYNX', 'WS ONIX', 'WS ORION', 'WS PEGASUS', 'WS PERSEUS',
    'WS PHOENIX', 'WS POLARIS', 'WS PROCYON', 'WS RIGEL', 'WS ROSALVO',
    'WS SAGITARIUS', 'WS SCORPIUS', 'WS SIRIUS', 'WS TITAN', 'WS VIRGO', 'WS ZEUS'
]

col_velocidade = 'Velocidade__nós_' 

# Ajustando tipos (Vírgula para ponto e converte para decimal) e convertendo datas
df_alarmes = df_alarmes.withColumn(col_velocidade, F.regexp_replace(F.col(col_velocidade).cast(StringType()), ',', '.').cast(DoubleType()))
df_alarmes = df_alarmes.withColumn('Disparo', F.to_timestamp(F.col('Disparo')))
df_alarmes = df_alarmes.dropDuplicates()

# Filtrar APENAS as embarcações que serão analisadas
df_alarmes = df_alarmes.filter(F.col('Embarcação').isin(rebocadores_ws))

# Remover Outliers com base em Estatística (Desvio Padrão)
estatisticas = df_alarmes.filter(F.col('Tipo') == 'Velocidade').select(
    F.mean(col_velocidade).alias('media'),
    F.stddev(col_velocidade).alias('desvio')
).collect()[0]

media_vel = estatisticas['media']
desvio_vel = estatisticas['desvio']
limite_superior = media_vel + (3 * desvio_vel)

print(f"Média calculada: {media_vel:.2f} | Desvio: {desvio_vel:.2f} | Limite Superior: {limite_superior:.2f} nós")

df_alarmes = df_alarmes.filter(
    ~(F.col('Tipo') == 'Velocidade') | 
    (F.col(col_velocidade).isNull()) | 
    (F.col(col_velocidade) <= limite_superior)
)

# ==========================================
# SALVANDO NA CAMADA SILVER
# ==========================================
df_manobras.write.format("delta").mode("overwrite").saveAsTable("workspace.default.silver_manobras")
df_alarmes.write.format("delta").mode("overwrite").saveAsTable("workspace.default.silver_alarmes")

print("Sucesso! Os dados foram limpos e salvos na Camada Silver.")